# Observability Performance Cost

## Setup
Use the separate engineering environment in README.md. Read the backend primer first. These assignments use local services and your own files; no model/API calls are made by default. All records and keys are invented teaching examples. Keep the whole track folder. Edit the numbered TODOs in `submission.py`, then run `importlib.reload(impl)` before rerunning tests. Instructor implementations are not included in the student handout.

Use a two-pass workflow: attempt each small TODO using the contract, then consult the staged hints in HINTS.md if needed. Record how much help you used in your learning log. Keep your solution in your own Git branch or copy. Never replace it with the instructor implementation before attempting the exercise.

In [1]:
from pathlib import Path
import sys, importlib, json
HERE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "engineeringlab").is_dir() or (p / "labs/engineering/engineeringlab").is_dir())
TRACK = HERE if (HERE / "engineeringlab").is_dir() else HERE / "labs/engineering"
if str(TRACK) not in sys.path: sys.path.insert(0, str(TRACK))
RUN_EXERCISES = True
MODULE = "instructor.reference"
impl = importlib.import_module(MODULE)
from engineeringlab import checks
print("Implement submission.py, reload it, then enable exercise checks. Default execution is not a pass.")

Implement submission.py, reload it, then enable exercise checks. Default execution is not a pass.


## Goal
Record real HTTP measurements, aggregate failures honestly, and design an optimization experiment with authorization and freshness constraints. Prerequisites: Labs 14, 18–19 and chapter 13. Plan two sessions (3–4 hours, unpiloted).

The first measurement targets the health endpoint to isolate HTTP/process overhead. It does not measure RAG or model inference. Your submission repeats the method on the integrated retrieval/action path and labels stages separately. Traces should identify request, component, duration, status, and configuration version; exclude API keys and unnecessary document text.

## Steps
### 1. TODO 20.1 — calculate trustworthy summaries
Implement summarize(rows). Each row has success (boolean), latency_ms and cost (finite nonnegative numbers). Include failed requests in the attempted denominator and latency distribution. Return the fields listed in submission.py. p95 uses the nearest-rank convention; with n samples its zero-based index is ceil(0.95*n)-1. For no attempts, rates and percentiles are null. For no successes, cost per success is null.

The exact-value fixture below is invented and uses arbitrary cost units. It checks arithmetic, not API billing. Observed provider usage and the applicable rate schedule are required before reporting monetary model costs.

In [2]:
if RUN_EXERCISES:
    importlib.reload(impl)
    rows=[{"success":True,"latency_ms":2.,"cost":.3},{"success":False,"latency_ms":10.,"cost":.2}]
    print(impl.summarize(rows))

{'attempted': 2, 'successes': 1, 'success_rate': 0.5, 'p50_ms': 6.0, 'p95_ms': 10.0, 'total_cost': 0.5, 'cost_per_success': 0.5}


### 2. TODO 20.2 — isolate cache entries
Implement cache_key(tenant,resource,version) and validate its inputs. Include every listed field in the returned tuple. Then design a read-cache experiment: populate one tenant/version, update the source, and prove that a new version does not receive stale evidence. Do not cache a write operation as if it were a read. The key exercise alone does not implement a cache or prove correct invalidation in a deployed system.

In [3]:
if RUN_EXERCISES:
    importlib.reload(impl)
    checks.measurements(impl)
    print("Denominators, invalid times, empty runs, tenant and version keys checked")

Denominators, invalid times, empty runs, tenant and version keys checked


### 3. Measure a real local service
The following invokes actual HTTP requests, with a 3-second request timeout and bounded worker count. Each request creates its own client, so connection setup contributes to latency. It records zero model cost because this endpoint calls no model; it does not claim zero hosting cost. The helper launches and stops its own temporary service, independently of your Compose project.

In [4]:
if RUN_EXERCISES:
    from engineeringlab.harness import running_service
    from engineeringlab.bench import measure
    with running_service(MODULE) as url:
        experiments=[]
        for workers in (1,2,4):
            rows=measure(url,n=20,workers=workers)
            experiments.append({"workers":workers,**impl.summarize(rows)})
    for row in experiments:print(row)
    print("Observed local HTTP results; no causal scaling conclusion from this single ordered run.")

{'workers': 1, 'attempted': 20, 'successes': 20, 'success_rate': 1.0, 'p50_ms': 7.129562494810671, 'p95_ms': 8.271665981737897, 'total_cost': 0.0, 'cost_per_success': 0.0}
{'workers': 2, 'attempted': 20, 'successes': 20, 'success_rate': 1.0, 'p50_ms': 8.763583493418992, 'p95_ms': 12.212790985358879, 'total_cost': 0.0, 'cost_per_success': 0.0}
{'workers': 4, 'attempted': 20, 'successes': 20, 'success_rate': 1.0, 'p50_ms': 8.78487499721814, 'p95_ms': 15.467916993657127, 'total_cost': 0.0, 'cost_per_success': 0.0}
Observed local HTTP results; no causal scaling conclusion from this single ordered run.


### 4. Design and run your comparison
Measure your Compose service and your Lab 16 retrieval endpoint. Keep workload, data, package/model versions, and final answer contract fixed. Separate warm-up and measured requests, change run order, and repeat conditions. Compare one change at a time: candidate count, eligible-read caching, connection reuse, or independent read parallelism. Retain failures and quality checks.

Optional live extension: use your own Gemini account, declare a spending limit before execution, record actual usage and failures, and compare against a simple baseline. A request count is not a monetary guarantee. Never report the fixture cost units above as dollars. GPU access is unnecessary for the required HTTP experiments.

### 4. TODO 20.3 — inspect an actual request trace
Implement trace_event, then start the service with COURSE_TRACE=1. The supplied middleware measures requests and emits your allowlisted JSON event to the Uvicorn log. Request bodies, keys, and URL parameter values are excluded. Send an approved request and a denied request. Match the X-Request-ID response header to each logged event and explain its HTTP status. Save the JSON event lines as JSONL. The route field is a template, not a raw URL.

Add a test passing a secret inside an extra headers argument and verify it is absent from the returned event. Run tests/test_engineering.py after implementing the function. These are local structured request logs; distributed OpenTelemetry spans and an external collector are optional extensions. Do not describe this exercise as distributed tracing.

## Checks and submission
Submit raw JSONL observations, a configuration manifest, a results table (attempts, successes, p50/p95, timeouts, cost units), and a short explanation. Plot distributions if your sample supports it. Include at least one failed request and explain its denominator treatment. Define your acceptance criteria before comparing configurations. No speedup is required for full credit; a sound negative result is valid.

Rubric: aggregation 20, cache identity/freshness experiment 15, safe request traces 15, actual measured comparison 30, interpretation/reproducibility 20. A 20-request exercise is practice, not a production capacity claim.

## Next Steps
Complete the course capstone using the offline outcome or the advanced live-evidence outcome. Reference: [Python statistics](https://docs.python.org/3/library/statistics.html), [HTTPX timeouts](https://www.python-httpx.org/advanced/timeouts/), chapter 13 of the course textbook.